[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/05_ch6_classification_finetuning.ipynb)

# Chapter 6 — Fine-Tuning for Classification (Spam Detector)

**Build a Large Language Model (From Scratch)** · notebook 5 of 7

So far the model *generates* text. Now we repurpose it to *classify* text:
**is this SMS message spam or not?** This is **fine-tuning** — take a pretrained
model and adapt it to a specific task with a little labeled data.

The plan:

1. Load the **pretrained GPT-2 (124M)** as our starting point.
2. Prepare the **SMS Spam** dataset (balanced, tokenized, padded).
3. **Freeze** most of the model, replace the output head with a 2-class head, and
   unfreeze just the last transformer block.
4. Train on the last-token output and measure **accuracy**.
5. Use the fine-tuned model to classify new messages.

> **Why freeze most layers?** The pretrained model already understands English.
> We only need to teach a small "read-out" on top, which is fast and needs little
> data. Only the last block + final norm + new head are trained.

*Runs on CPU: base model loads in seconds; fine-tuning a few minutes. Much
faster on a Colab GPU.*

## 0. Setup

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, name=None):
    try: importlib.import_module(name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
for p in ["tiktoken", "pandas", "transformers"]:
    ensure(p)

import torch, torch.nn as nn, tiktoken, os, urllib.request, zipfile
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
print("torch:", torch.__version__, "| device:", device)

torch: 2.11.0+cu128 | device: cuda


## Recap: the GPT model (from Chapter 4)

Same model class, so this notebook is self-contained.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out, self.num_heads, self.head_dim = d_out, num_heads, d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, n, _ = x.shape
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(2, 3)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        w = self.dropout(torch.softmax(scores / k.shape[-1]**0.5, dim=-1))
        ctx = (w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(ctx)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)); self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True); var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715*torch.pow(x,3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]), GELU(),
                                    nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
                                      cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"]); self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x = x + self.drop_shortcut(self.att(self.norm1(x)))
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        b, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        return self.out_head(self.final_norm(self.trf_blocks(x)))

def _copy(dst, src):
    assert dst.shape == src.shape, f"{tuple(dst.shape)} vs {tuple(src.shape)}"
    with torch.no_grad(): dst.copy_(src)

def load_weights_from_hf(gpt, hf_model):
    sd = hf_model.state_dict()
    _copy(gpt.tok_emb.weight, sd["transformer.wte.weight"])
    _copy(gpt.pos_emb.weight, sd["transformer.wpe.weight"])
    for b in range(len(gpt.trf_blocks)):
        p, blk = f"transformer.h.{b}.", gpt.trf_blocks[b]
        w = sd[p+"attn.c_attn.weight"]; q,k,v = w.split(w.shape[1]//3, dim=1)
        _copy(blk.att.W_query.weight, q.T); _copy(blk.att.W_key.weight, k.T); _copy(blk.att.W_value.weight, v.T)
        bq,bk,bv = sd[p+"attn.c_attn.bias"].split(sd[p+"attn.c_attn.bias"].shape[0]//3, dim=0)
        _copy(blk.att.W_query.bias, bq); _copy(blk.att.W_key.bias, bk); _copy(blk.att.W_value.bias, bv)
        _copy(blk.att.out_proj.weight, sd[p+"attn.c_proj.weight"].T); _copy(blk.att.out_proj.bias, sd[p+"attn.c_proj.bias"])
        _copy(blk.norm1.scale, sd[p+"ln_1.weight"]); _copy(blk.norm1.shift, sd[p+"ln_1.bias"])
        _copy(blk.norm2.scale, sd[p+"ln_2.weight"]); _copy(blk.norm2.shift, sd[p+"ln_2.bias"])
        _copy(blk.ff.layers[0].weight, sd[p+"mlp.c_fc.weight"].T); _copy(blk.ff.layers[0].bias, sd[p+"mlp.c_fc.bias"])
        _copy(blk.ff.layers[2].weight, sd[p+"mlp.c_proj.weight"].T); _copy(blk.ff.layers[2].bias, sd[p+"mlp.c_proj.bias"])
    _copy(gpt.final_norm.scale, sd["transformer.ln_f.weight"]); _copy(gpt.final_norm.shift, sd["transformer.ln_f.bias"])
    _copy(gpt.out_head.weight, sd["transformer.wte.weight"])

print("Model code ready.")

Model code ready.


## 6.1 Load the pretrained GPT-2 (124M)

This is our starting point — an English-savvy model. If the download is
unavailable, we fall back to a small randomly-initialized model so the notebook
still runs (accuracy will be lower, but the *pipeline* is identical).

In [ ]:
GPT2_124M = {"vocab_size":50257, "context_length":1024, "emb_dim":768,
             "n_heads":12, "n_layers":12, "drop_rate":0.0, "qkv_bias":True}

import os, warnings
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore")

model = None
try:
    from transformers import GPT2LMHeadModel
    try:
        from transformers.utils import logging as hf_logging
        hf_logging.disable_progress_bar(); hf_logging.set_verbosity_error()
    except Exception:
        pass
    print("Loading pretrained GPT-2 124M ...")
    hf_model = GPT2LMHeadModel.from_pretrained("gpt2"); hf_model.eval()
    model = GPTModel(GPT2_124M)
    load_weights_from_hf(model, hf_model)
    print("Pretrained GPT-2 loaded.")
except Exception as e:
    print("Pretrained load failed:", repr(e), "\nFalling back to a small model.")
    GPT2_124M = {**GPT2_124M, "emb_dim":256, "n_heads":8, "n_layers":6}
    torch.manual_seed(123); model = GPTModel(GPT2_124M)
model.to(device)
print("Base model ready with emb_dim =", GPT2_124M["emb_dim"])

Loading pretrained GPT-2 124M ...


Pretrained GPT-2 loaded.
Base model ready with emb_dim = 768


## 6.2 Preparing the dataset

We use the **SMS Spam Collection** (5,574 real messages labeled ham/spam). We
download it, **balance** it (equal ham/spam so accuracy is meaningful), map
labels to 0/1, and split into train/validation/test. If the download fails, a
small built-in sample is used instead.

In [ ]:
def load_spam_dataframe():
    url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
    zip_path, extracted = "sms_spam_collection.zip", "sms_spam_collection"
    tsv_path = Path(extracted) / "SMSSpamCollection.tsv"
    try:
        if not tsv_path.exists():
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(extracted)
            os.rename(Path(extracted) / "SMSSpamCollection", tsv_path)
        df = pd.read_csv(tsv_path, sep="\t", header=None, names=["Label", "Text"])
        print("Loaded real SMS Spam Collection:", df.shape)
        return df
    except Exception as e:
        print("Download failed:", repr(e), "-- using a small built-in sample.")
        ham = ["Are we still on for lunch today?", "Can you send me the meeting notes?",
               "I'll be home around 8, see you soon", "Thanks for your help yesterday!",
               "Don't forget to pick up milk", "Happy birthday! Have a great day",
               "The report is ready for your review", "Let's catch up over coffee this week",
               "Running 10 mins late, sorry", "Did you watch the game last night?"]
        spam = ["WINNER!! You have won a $1000 gift card. Call now to claim!",
                "URGENT! Your account is suspended. Click http://bit.ly/x to verify",
                "Free entry to WIN tickets! Text WIN to 80085 now",
                "Congratulations! You've been selected for a free cruise",
                "Claim your FREE ringtone now, reply YES to 55555",
                "You have 1 new voicemail. Call 09061701461 to listen (premium rate)",
                "Get cheap loans fast! No credit check. Apply today",
                "Hot singles in your area want to meet you! Click here",
                "Final notice: pay now to avoid account closure. Click link",
                "You won a lottery of $5,000,000! Send bank details to claim"]
        rows = [("ham", t) for t in ham*8] + [("spam", t) for t in spam*8]
        return pd.DataFrame(rows, columns=["Label", "Text"])

df = load_spam_dataframe()
print(df["Label"].value_counts().to_dict())

Loaded real SMS Spam Collection: (5572, 2)
{'ham': 4825, 'spam': 747}


In [ ]:
def create_balanced_dataset(df):
    num_spam = (df["Label"] == "spam").sum()
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    return pd.concat([ham_subset, df[df["Label"] == "spam"]])

balanced = create_balanced_dataset(df)
balanced["Label"] = balanced["Label"].map({"ham": 0, "spam": 1})

def random_split(df, train_frac, val_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    n_train = int(len(df) * train_frac)
    n_val = int(len(df) * val_frac)
    return df[:n_train], df[n_train:n_train+n_val], df[n_train+n_val:]

train_df, val_df, test_df = random_split(balanced, 0.7, 0.1)

# On CPU we cap training examples so it finishes fast; on GPU we use ALL of them.
TRAIN_SUBSET = None if device.type == "cuda" else 700
if TRAIN_SUBSET is not None and len(train_df) > TRAIN_SUBSET:
    train_df = train_df.iloc[:TRAIN_SUBSET]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(f"{name}.csv", index=None)
print("Split sizes -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))

Split sizes -> train: 1045 val: 149 test: 300


## 6.3 Creating data loaders

Each message is tokenized and **padded** to a common length so we can batch them.
We use the `<|endoftext|>` id (50256) as the padding token.

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        self.encoded_texts = [tokenizer.encode(t) for t in self.data["Text"]]
        if max_length is None:
            self.max_length = max(len(e) for e in self.encoded_texts)
        else:
            self.max_length = max_length
            self.encoded_texts = [e[:max_length] for e in self.encoded_texts]
        self.encoded_texts = [
            e + [pad_token_id] * (self.max_length - len(e)) for e in self.encoded_texts]
    def __getitem__(self, i):
        return (torch.tensor(self.encoded_texts[i], dtype=torch.long),
                torch.tensor(self.data.iloc[i]["Label"], dtype=torch.long))
    def __len__(self): return len(self.data)

# Cap length to keep CPU training snappy (SMS messages are short anyway).
MAX_LEN = min(256, SpamDataset("train.csv", tokenizer).max_length)
train_ds = SpamDataset("train.csv", tokenizer, max_length=MAX_LEN)
val_ds   = SpamDataset("val.csv",   tokenizer, max_length=MAX_LEN)
test_ds  = SpamDataset("test.csv",  tokenizer, max_length=MAX_LEN)
print("Padded sequence length:", MAX_LEN)

bs = 8
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, drop_last=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, drop_last=False, num_workers=0)
print("Train batches:", len(train_loader))

Padded sequence length: 120
Train batches: 130


## 6.4 & 6.5 Freeze the model and add a classification head

We freeze every parameter, then:
- Replace `out_head` (which output 50,257 vocab logits) with a small
  `Linear(emb_dim, 2)` head — spam vs not-spam.
- Unfreeze the **last transformer block** and the **final layer norm** so the
  model can adapt its top-level representation to the task.

In [ ]:
# freeze everything
for p in model.parameters():
    p.requires_grad = False

# new 2-class head (created with requires_grad=True by default)
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(GPT2_124M["emb_dim"], num_classes)

# unfreeze the last block + final norm
for p in model.trf_blocks[-1].parameters():
    p.requires_grad = True
for p in model.final_norm.parameters():
    p.requires_grad = True

model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

Trainable parameters: 7,090,946


## 6.6 Classification loss and accuracy

For classification we look only at the model's output **at the last token
position** — that vector summarizes the whole message. We put the 2-class head on
it and use cross-entropy.

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct, total = 0, 0
    n = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (inp, tgt) in enumerate(data_loader):
        if i >= n: break
        inp, tgt = inp.to(device), tgt.to(device)
        with torch.no_grad():
            logits = model(inp)[:, -1, :]          # last-token logits
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == tgt).sum().item(); total += tgt.numel()
    return correct / total

def calc_loss_batch(inp, tgt, model, device):
    inp, tgt = inp.to(device), tgt.to(device)
    logits = model(inp)[:, -1, :]
    return torch.nn.functional.cross_entropy(logits, tgt)

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total, n = 0.0, (len(data_loader) if num_batches is None else min(num_batches, len(data_loader)))
    if n == 0: return float("nan")
    for i, (inp, tgt) in enumerate(data_loader):
        if i >= n: break
        total += calc_loss_batch(inp, tgt, model, device).item()
    return total / n

print("Accuracy BEFORE fine-tuning (should be ~50%, i.e. random):",
      round(calc_accuracy_loader(val_loader, model, device, num_batches=10) * 100, 1), "%")

Accuracy BEFORE fine-tuning (should be ~50%, i.e. random): 45.0 %


## 6.7 Fine-tuning the model

Now train for **2 epochs** on a subset of the data. On a laptop CPU this takes a
few minutes (each step runs a forward pass through all 12 layers of GPT-2). On a
Colab GPU it's seconds — there you can set `TRAIN_SUBSET = None` and train longer
for even higher accuracy.

In [ ]:
import time

def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for inp, tgt in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(inp, tgt, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += inp.shape[0]; global_step += 1
            if global_step % eval_freq == 0:
                tr = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                vl = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
                train_losses.append(tr); val_losses.append(vl)
                print(f"Ep {epoch+1} (step {global_step:04d}): train loss {tr:.3f}, val loss {vl:.3f}")
        ta = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        va = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"  -> epoch {epoch+1}: train acc {ta*100:.1f}% | val acc {va*100:.1f}%")
        train_accs.append(ta); val_accs.append(va)
    return train_losses, val_losses, train_accs, val_accs

torch.manual_seed(123)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=2e-4, weight_decay=0.1)
NUM_EPOCHS = 5 if device.type == "cuda" else 2   # more epochs on GPU
t0 = time.time()
train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                        num_epochs=NUM_EPOCHS, eval_freq=50, eval_iter=5)
print(f"Fine-tuning took {time.time()-t0:.1f}s on {device}")

Ep 1 (step 0000): train loss 1.618, val loss 1.796


Ep 1 (step 0050): train loss 0.521, val loss 0.553


Ep 1 (step 0100): train loss 0.310, val loss 0.428


  -> epoch 1: train acc 87.5% | val acc 85.0%


Ep 2 (step 0150): train loss 0.720, val loss 0.443


...

Ep 4 (step 0500): train loss 0.177, val loss 0.018


  -> epoch 4: train acc 100.0% | val acc 95.0%


Ep 5 (step 0550): train loss 0.133, val loss 0.046


Ep 5 (step 0600): train loss 0.035, val loss 0.013


  -> epoch 5: train acc 100.0% | val acc 100.0%
Fine-tuning took 16.5s on cuda


In [ ]:
# Final accuracy on the held-out test set
test_acc = calc_accuracy_loader(test_loader, model, device)
print(f"Test accuracy: {test_acc*100:.1f}%")

Test accuracy: 97.0%


## 6.8 Using the LLM as a spam classifier

A helper to classify any new message.

In [ ]:
def classify_review(text, model, tokenizer, device, max_length, pad_token_id=50256):
    model.eval()
    ids = tokenizer.encode(text)[:max_length]
    ids += [pad_token_id] * (max_length - len(ids))
    inp = torch.tensor(ids, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(inp)[:, -1, :]
    return "spam" if torch.argmax(logits, dim=-1).item() == 1 else "not spam"

tests = [
    "Congratulations! You have WON a free entry to claim a 2000 cash prize. Call 09061234567 now!",
    "Hey, are we still meeting for dinner tonight?",
    "URGENT! Your mobile number has been awarded a 1000 prize. Txt WIN to 85233 to claim now",
    "Can you review my pull request when you get a chance?",
]
for t in tests:
    print(f"[{classify_review(t, model, tokenizer, device, MAX_LEN):>8}]  {t[:60]}")

[    spam]  Congratulations! You have WON a free entry to claim a 2000 c
[not spam]  Hey, are we still meeting for dinner tonight?
[    spam]  URGENT! Your mobile number has been awarded a 1000 prize. Tx
[not spam]  Can you review my pull request when you get a chance?


## Summary

You fine-tuned a **generative** model into a **classifier** by:

- Loading pretrained GPT-2 as an English-understanding backbone.
- Freezing it and attaching a small 2-class head on the **last-token** output.
- Training only the top layers on labeled data — fast, data-efficient transfer
  learning.

### Try it yourself
- Increase `num_epochs` to 3-5 and watch test accuracy climb.
- Unfreeze more blocks (`model.trf_blocks[-2]`) and compare.
- Feed your own messages to `classify_review`.

**Next:** `06_ch7_instruction_finetuning.ipynb` — teach the model to *follow
instructions*, the technique behind chat assistants.